# Artifacts and jobs: load, resolve, run

A walkthrough of the current artifact/job machinery, entirely local (no Modal
involved).

The shape every family follows: `artifacts/<family>/__init__.py` defines the
artifact class(es) (`from artifacts.sources import Source`); a sibling
`artifacts/<family>/jobs.py` defines the job(s) that produce them
(`from artifacts.sources.jobs import SourceJob`). An artifact never imports
its own job module -- it only *names* it:

- `artifact.producer` -- a hardcoded dotted path, e.g.
  `"artifacts.sources.jobs.SourceJob"`. A plain string, free to read.
- `artifact.job()` -- resolves that string into the real `Job` subclass
  through `locate()` and constructs it. The one call that actually imports
  a `jobs.py`, and with it whatever torch or numpy that family carries.

That split is what the rest of this notebook rests on. Resolving a graph,
checking status, planning and drawing all work in artifacts and paths and
never touch a job. `job()` is reached once, at the very end, at the point of
actually running something.

In [ ]:
import json
import logging
import shutil
from pathlib import Path
from types import SimpleNamespace

from artifacts.core import resolve as core_resolve
from artifacts.core.artifact import Artifact
from artifacts.mappeddataset import MappedDataSet
from artifacts.sources import Source
from artifacts.tokenizers.bpe import Tokenizer

# a throwaway local "volume" -- everything below writes only here
ROOT = Path(".scratch/resolve-demo").resolve()
shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
# The minimal stand-in for system.runtime.Worker every job.run(root, worker)
# needs. A real Worker also carries a lease, which nothing here touches --
# but `progress` has to be here: a job reports incremental progress by
# calling worker.progress.update(...) (TokenizerJob does, per merge), and a
# plain dict takes that the same way the real one does. Nothing reads it
# back here; on the volume the dashboard reads it off the call's heartbeat.
worker = SimpleNamespace(log=logging.getLogger("demo"), progress={})

ROOT

## Declare a small tree

`MappedDataSet.from_sources` builds a `TokenizedSource` per source under the
hood -- one artifact per (tokenizer, source) pair. Nothing here is written
yet; constructing an artifact is just naming its parameters.

In [ ]:
romeojuliet = Source(name="romeojuliet", url="https://www.gutenberg.org/cache/epub/1513/pg1513.txt")
mobydick = Source(name="mobydick", url="https://www.gutenberg.org/cache/epub/2701/pg2701.txt")

tokenizer = Tokenizer(vocab_size=500, special_tokens=("<|endoftext|>",), sources=(romeojuliet, mobydick))

mapped = MappedDataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=[romeojuliet, mobydick],
    valid_sources=[romeojuliet],
)

print("uid:          ", mapped.uid)
print("artifact_path:", mapped.artifact_path)
print("producer:     ", mapped.producer)  # a dotted path, nothing imported

## Loading: manifest <-> object

`manifest()` is the artifact as JSON -- what actually lands in
`manifest.json`. `"artifact"` is a fully-qualified dotted path to the class
(`artifacts.mappeddataset.MappedDataSet`), not just a bare name: `locate()`
resolves it directly, so nothing needs pre-importing every family just to
read a manifest back, and two families can reuse the same class name
without colliding.

`Artifact.from_manifest` is the inverse -- `from_manifest(a.manifest()) == a`
for every artifact, always. `Artifact.load(path)` is that same round trip
read straight off a `manifest.json` file on disk.

In [3]:
manifest = mapped.manifest()
print(json.dumps(manifest, indent=2)[:600], "...")

{
  "artifact": "artifacts.mappeddataset.MappedDataSet",
  "commit": "be370b8a3cd4a28555ed57a093840ed943e33ec6-dirty",
  "allocated_resources": {
    "cpu": null,
    "gpu_count": null,
    "gpu_type": null
  },
  "parameters": {},
  "dependencies": {
    "train_set": [
      {
        "artifact": "artifacts.tokenizers.bpe.TokenizedSource",
        "commit": "be370b8a3cd4a28555ed57a093840ed943e33ec6-dirty",
        "allocated_resources": {
          "cpu": null,
          "gpu_count": null,
          "gpu_type": null
        },
        "parameters": {},
        "dependencies": {
          "sou ...


In [4]:
rebuilt = Artifact.from_manifest(manifest)
assert rebuilt == mapped
print("from_manifest(a.manifest()) == a:", rebuilt == mapped)

from_manifest(a.manifest()) == a: True


## Resolving: building the graph

`resolve(artifact)` walks `artifact.deps()` recursively and returns a `Dag`:
every artifact reachable from the request, deduplicated by `artifact_path`,
with the paths of what each one depends on.

No job appears in it. A job produces exactly one artifact, so a parallel list
of jobs would be the same list under different names -- bought at the price of
importing every family in the graph.

One walk does all of it. A path already resolved is returned immediately, so a
shared `Source` reached through two `TokenizedSource`s is visited once, not
twice. And a node is recorded only after everything it depends on, which makes
the node map's own order a topological sort.

In [ ]:
dag = core_resolve.resolve(mapped)

print("nodes:      ", len(dag))
print("roots:      ", [str(p) for p in dag.roots])
print("target:     ", dag.target, "(no target -> nothing was read from disk)")
print("mapped deps:", [p.name for p in dag[mapped.artifact_path].deps])

## The plan: dependency order, deduplicated

`plan(dag)` is the artifacts to build, in the order they have to be built.
Because one job produces one artifact, that list *is* the work: the nth entry
is the nth thing to run.

`resolve` already ordered and deduplicated the graph, so `plan` doesn't sort --
what it does is select. `pending_only=True` drops whatever already reads
`done`, which is what you want right before running something.

In [ ]:
for i, artifact in enumerate(core_resolve.plan(dag), 1):
    print(f"{i}. {type(artifact).__name__:16} {artifact.artifact_path}")

## Declare: write the manifests

Pass a `target` and the same walk also checks each artifact against that root,
storing the answer on its node -- so resolving *is* the reconciliation, not a
separate pass over it. Everything reads `new` here, since nothing has been
declared in `ROOT` yet.

`declare(dag)` then writes one `manifest.json` per `new` artifact,
dependencies first, refusing outright if anything in the graph is
inconsistent.

In [ ]:
dag = core_resolve.resolve(mapped, target=ROOT)
print(dag)

In [ ]:
core_resolve.declare(dag)

# `declare` leaves the graph as it was resolved -- a snapshot from before the
# write, where everything just written still reads `new`. Resolve again to see
# what is actually on disk now.
dag = core_resolve.resolve(mapped, target=ROOT)
print(dag)

## Running: the one place a job is constructed

`plan(dag, pending_only=True)` gives the artifacts still to build, already in
the right order. `artifact.job()` turns each one's `producer` string into a
real `Job` -- the first and only time this notebook imports a `jobs.py`.

`MappedDataSet` is the interesting case. It owns no files of its own and reads
`done` the instant its sources are tokenized, so it is normally filtered out
before `job()` is ever reached for it. Not here: this graph was resolved before
anything existed, so it was still `declared` when the plan was taken and its
job does get constructed and run. That body is a deliberate no-op -- exactly
the "plan built and run before its sources exist yet" case
`MappedDataSetJob`'s own docstring describes.

In [ ]:
for artifact in core_resolve.plan(dag, pending_only=True):
    print(f"running {artifact.producer} for {artifact.artifact_path}")
    artifact.job().run(ROOT, worker)

print()
print(core_resolve.resolve(mapped, target=ROOT))

## `producer` vs `job()`: a string vs a constructed instance

`producer` is the full dotted path, hardcoded on the artifact class -- a plain
string, never an import. `job()` is what resolves and constructs it, through
`locate()`. They always agree: the class `job()` builds is exactly the one
`producer` names.

Written out in full rather than derived from `type(self).__module__` plus a
bare name, so which job produces a given artifact is answerable by reading the
artifact class, with nothing to compute.

In [ ]:
job_instance = tokenizer.job()

print("tokenizer.producer:  ", tokenizer.producer)
print("type(tokenizer.job()):", f"{type(job_instance).__module__}.{type(job_instance).__qualname__}")
assert f"{type(job_instance).__module__}.{type(job_instance).__qualname__}" == tokenizer.producer

## Reading back what was declared

`bind(root)` returns a *new*, bound copy with whatever a job wrote read back
in -- `mapped`'s `train_tokens`/`valid_tokens` (a memmap per source), and
the tokenizer's vocab/merges. The convention is to reassign
(`mapped = mapped.bind(ROOT)`), since the object `bind` was called on is
left exactly as unbound as it was.

In [11]:
mapped = mapped.bind(ROOT)
bound_tokenizer = tokenizer.bind(ROOT)

print(f"{len(mapped.train_tokens)} train tokens across {len(mapped.train_set)} source(s)")
print(repr(bound_tokenizer.decode(mapped.train_tokens[:15])))

706901 train tokens across 2 source(s)
'The Project Gutenberg eBook'


In [12]:
# Cleanup, if you want the demo folder gone:
# import shutil; shutil.rmtree(ROOT)